# EDGAR Financial Sentiment — Part 7: Structured Extraction (JSON + Chain-of-Thought)

The last Track B part. So far every method produced **one label** per text. Here the LLM returns a **structured record** — sentiment *and* guidance direction *and* key figures *and* a rationale — as JSON, reasoning step by step first. This is the on-ramp to real **8-K earnings releases**.

**What you'll practice (the core concept):** **structured-output prompting** (`build_extraction_prompt`) and **robust JSON parsing** (`parse_json`) — those are your blanks. You'll then *measure* whether chain-of-thought + a schema actually help, against a minimal prompt.

> **Run in Google Colab with a T4 GPU.** Uses the gated Mistral model (Parts 4–6 login). Prompts are original and finance-specific (kept separate from the ungraded Assignment 4).

## 0. Why structured extraction? From a label to a record

A single sentiment label throws away most of what a financial statement says. An earnings release tells you **which way guidance moved**, **what EPS and revenue were**, and **why** — a whole record, not a mood. **Structured extraction** asks the model to return that record as **JSON** instead of a word:

```json
{"sentiment": "positive", "guidance_direction": "raised",
 "key_figures": ["EPS $1.42", "revenue $2.8B"], "rationale": "..."}
```

That's something you can put in a database, regress on, or chart — and it's the on-ramp to **real 8-K EX-99.1 earnings releases** (the data pipeline in `PROJECT_SCOPE.md`).

**Two new skills + one technique:**
- **Structured-output prompting** — getting the model to emit a *parseable* object that follows a schema.
- **Robust JSON parsing** — LLMs wrap JSON in prose, markdown fences, or trailing commas; your parser must survive that and fail gracefully (return `None`) when there's no JSON.
- **Chain-of-thought (CoT)** — "reason first, then answer" usually improves the answer. We'll *measure* whether it does.

**What we measure:** the **validity rate** (did we get parseable JSON?) and agreement with a small hand-labeled gold set — your CoT+schema prompt vs. a minimal one. You'll also see that **objective** fields (guidance direction, figures) extract far more reliably than the **subjective** overall sentiment — a real, useful finding.

## 1. Setup

In [71]:
!pip install -q -U transformers bitsandbytes accelerate

In [72]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [73]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

Device: cuda


### Hugging Face login (Mistral is gated)
Same as Parts 4–6: accept the license, make a token, log in.

In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 2. Config & the target schema

In [74]:
torch.manual_seed(10)

MODEL_ID   = 'mistralai/Mistral-7B-Instruct-v0.3'

# The schema we want the model to fill, and the allowed values for the two we can score:
SENTIMENTS = ['positive', 'neutral', 'negative']
GUIDANCE   = ['raised', 'lowered', 'maintained', 'none']

## 3. The data — press-release excerpts + a small gold set  *(provided)*

Six short, realistic earnings-statement excerpts, each hand-labeled with the *overall sentiment* and the *guidance direction* so we can score the extraction. (These are press-release-**style** text; the real target is 8-K EX-99.1 filings — see Section 9.)

In [75]:
EXCERPTS = [
    {'text': 'Helios Semiconductor today reported third-quarter diluted EPS of $1.42, ahead of its prior '
             'outlook, on revenue of $2.8 billion, up 18% year over year. Citing strong demand for its '
             'data-center products, the company raised its full-year revenue guidance to a range of $11.2 '
             'to $11.5 billion.',
     'gold': {'sentiment': 'positive', 'guidance_direction': 'raised'}},
    {'text': 'Meridian Retail Group reported a fourth-quarter net loss of $0.31 per share as comparable-store '
             'sales fell 6%. Management pointed to soft consumer spending and elevated markdowns, and lowered '
             'its full-year earnings guidance to $0.90-$1.05 per share from the prior $1.30-$1.45.',
     'gold': {'sentiment': 'negative', 'guidance_direction': 'lowered'}},
    {'text': 'Cascade Utilities announced second-quarter earnings of $0.88 per share, in line with analyst '
             'expectations, on revenue of $1.1 billion. The company reaffirmed its full-year guidance of '
             '$3.40 to $3.55 per share.',
     'gold': {'sentiment': 'neutral', 'guidance_direction': 'maintained'}},
    {'text': 'Orion Biotech delivered first-quarter revenue of $410 million, beating consensus, and EPS of '
             '$0.22 versus a loss a year ago. However, management cautioned that pricing pressure would weigh '
             'on the back half and trimmed its full-year operating margin outlook.',
     'gold': {'sentiment': 'negative', 'guidance_direction': 'lowered'}},   # mixed: a beat, but a cut outlook
    {'text': 'Granite Industrial said it completed the acquisition of Pinnacle Tooling for $640 million in '
             'cash, expanding its presence in precision components. The transaction closed in the fourth '
             'quarter.',
     'gold': {'sentiment': 'neutral', 'guidance_direction': 'none'}},
    {'text': 'Aurora Software reported record quarterly bookings and EPS of $0.67, up from $0.51, driven by '
             '30% growth in its cloud segment. The company did not update its annual forecast.',
     'gold': {'sentiment': 'positive', 'guidance_direction': 'none'}},
]
print(len(EXCERPTS), 'excerpts loaded')

6 excerpts loaded


## 4. Load Mistral-7B (4-bit) + a generation helper  *(provided)*

Causal LM, same as Parts 5–6. Note `generate` is **greedy** (`do_sample=False`) here — for extraction we want deterministic, reproducible output, not the diversity we wanted when *generating* data in Part 6.

In [76]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={'': 0})
model.eval()
print('Loaded', MODEL_ID)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded mistralai/Mistral-7B-Instruct-v0.3


In [82]:
@torch.no_grad()
def generate(prompt_text, max_new_tokens=256):
    messages = [{'role': 'user', 'content': prompt_text}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(device)
    out = model.generate(inputs['input_ids'], max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

## 5. Write the extraction prompt  &larr; **your code**

`build_extraction_prompt(text)` should ask the model to **reason briefly, then emit one JSON object** with these keys:
- `"sentiment"` — one of `positive` / `neutral` / `negative`
- `"guidance_direction"` — one of `raised` / `lowered` / `maintained` / `none`
- `"key_figures"` — a list of short strings like `"EPS $1.42"`
- `"rationale"` — a one-sentence justification

Good structured prompts: **name every key and its allowed values**, **show the JSON shape**, ask for **valid JSON only** (double quotes, no trailing commas), and put the chain-of-thought *before* the JSON. The clearer the format demand, the easier your `parse_json` has it.

In [83]:
def build_extraction_prompt(text):
    """Reason step by step, then emit one JSON object following the schema."""
    ### YOUR CODE HERE
    # Return one instruction string that:
    #  - tells the model to think step by step FIRST, then output a SINGLE JSON object
    #  - names all four keys and their allowed values:
    #      sentiment        -> positive / neutral / negative
    #      guidance_direction -> raised / lowered / maintained / none
    #      key_figures      -> list of short strings (e.g. "EPS $1.42")
    #      rationale        -> one sentence
    #  - demands valid JSON only (double quotes, no trailing commas)
    #  - then appends the statement text (e.g. ... + '\n\nStatement:\n' + text)
    return "\n".join([
        "You are a Financial Analyst who must interpret a Statement from company management and write a report.",
        "Read the Statement at the end and thing step by step about the answers in your report.",
        "You must deliver a valid JSON format object with the following 4 keys.",
        "The 4 keys are provided as:",
        "'key': [valid response] (definition)",
        "{",
        "'sentiment': [positive / neutral / negative] (the overall sentiment of the Statement)",
        "'guidance_direction': [raised / lowered / maintained / none] (management's outlook for the company)",
        "'key_figures': [list of short strings] (citations directly from the Statement)",
        "'rationale': [one sentence] (explain your step by step thinking based on the figures to explain the sentiment and guidance_direction)",
        "}",
        "\n\nStatement:\n",
    ]) + text
    ### END YOUR CODE

### See your prompt + a raw reply
Run after filling in `build_extraction_prompt`. Notice the model reasons, *then* writes JSON — your parser has to find the JSON inside that.

In [84]:
print('=== PROMPT ===')
print(build_extraction_prompt(EXCERPTS[0]['text']))
print('\n=== MODEL REPLY ===')
print(generate(build_extraction_prompt(EXCERPTS[0]['text'])))

=== PROMPT ===
You are a Financial Analyst who must interpret a Statement from company management and write a report.
Read the Statement at the end and thing step by step about the answers in your report.
You must deliver a valid JSON format object with the following 4 keys.
The 4 keys are provided as:
'key': [valid response] (definition)
{
'sentiment': [positive / neutral / negative] (the overall sentiment of the Statement)
'guidance_direction': [raised / lowered / maintained / none] (management's outlook for the company)
'key_figures': [list of short strings] (citations directly from the Statement)
'rationale': [one sentence] (explain your step by step thinking based on the figures to explain the sentiment and guidance_direction)
}


Statement:
Helios Semiconductor today reported third-quarter diluted EPS of $1.42, ahead of its prior outlook, on revenue of $2.8 billion, up 18% year over year. Citing strong demand for its data-center products, the company raised its full-year revenue 

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
"sentiment": "positive",
"guidance_direction": "raised",
"key_figures": ["$1.42", "$2.8 billion", "$11.2 to $11.5 billion"],
"rationale": "The positive sentiment is derived from the company exceeding its prior EPS outlook and reporting a 18% YoY revenue growth. The guidance direction is 'raised' as the company has increased its full-year revenue guidance due to strong demand for its data-center products."
}


## 6. Parse the JSON robustly  &larr; **your code**

`parse_json(reply)` returns a **dict**, or **`None`** if there's no usable JSON. The model's reply has reasoning prose (and sometimes ```` ```json ```` fences) around the object, so you can't just `json.loads` the whole thing. A robust trick: take everything from the **first `{`** to the **last `}`** and try to parse that; return `None` if it fails. Returning `None` (instead of crashing) is what lets you *count* failures as the validity rate.

In [ ]:
def parse_json(reply):
    """Extract the JSON object from the reply; return a dict, or None if unparseable."""
    ### YOUR CODE HERE
    # 1. find the first '{' and the last '}' in reply
    # 2. if either is missing (or out of order), return None
    # 3. try json.loads on that substring; return the dict, or None on json.JSONDecodeError
    try:
      valid_json = json.loads(reply[reply.index('{'):reply.rindex('}')+1])
    except json.JSONDecodeError:
      return None
    return valid_json

    ### END YOUR CODE

### Test your parser
It should pull the dict out of the prose, and return `None` when there's no JSON.

In [79]:
_messy = 'Here is the result: {"sentiment": "positive", "guidance_direction": "raised", '\
         '"key_figures": ["EPS $1.42"]} -- hope that helps!'
print(parse_json(_messy))                                  # -> dict
print(parse_json('the model only wrote prose, no JSON'))   # -> None

{'sentiment': 'positive', 'guidance_direction': 'raised', 'key_figures': ['EPS $1.42']}
None


## 7. Extract from every excerpt  *(provided driver)*
Run your prompt + parser over all six and read the structured output. This is the qualitative look before we score.

In [85]:
for ex in EXCERPTS:
    obj = parse_json(generate(build_extraction_prompt(ex['text'])))
    print('TEXT:', ex['text'][:85], '...')
    print('EXTRACTED:', json.dumps(obj, indent=2) if obj else '<<invalid / no JSON>>')
    print('GOLD:', ex['gold'], '\n' + '-' * 70)

TEXT: Helios Semiconductor today reported third-quarter diluted EPS of $1.42, ahead of its  ...
EXTRACTED: {
  "sentiment": "positive",
  "guidance_direction": "raised",
  "key_figures": [
    "$1.42",
    "$2.8 billion",
    "$11.2 to $11.5 billion"
  ],
  "rationale": "The positive sentiment is derived from the company exceeding its prior EPS outlook and reporting a 18% YoY revenue growth. The guidance direction is 'raised' as the company has increased its full-year revenue guidance due to strong demand for its data-center products."
}
GOLD: {'sentiment': 'positive', 'guidance_direction': 'raised'} 
----------------------------------------------------------------------
TEXT: Meridian Retail Group reported a fourth-quarter net loss of $0.31 per share as compar ...
EXTRACTED: {
  "sentiment": "negative",
  "guidance_direction": "lowered",
  "key_figures": [
    "$0.31 per share net loss",
    "6% decrease in comparable-store sales",
    "$0.90-$1.05 per share full-year earnings guidanc

#### ✍️ Reflect — read the records
- Did `key_figures` and `guidance_direction` come out right? Were they easier than `sentiment`? _…_
- Look at the mixed excerpt (Orion: beat earnings, cut outlook) — what sentiment did the model pick, and do you agree? _…_
- Was every reply valid JSON? If not, what did the bad one look like? _…_

## 8. Measure — does CoT + a schema actually help?

#### ✍️ Predict before you run
- Your prompt's valid-JSON rate (out of 6): _…_  Minimal prompt's: _…_
- Which field will the model get right more often — `sentiment` or `guidance_direction`? Why? _…_
- Why might a minimal prompt score badly even if the model 'knows' the answer? _…_

In [ ]:
def build_extraction_prompt_minimal(text):
    """A terse contrast prompt: no step-by-step, no schema, no format demand."""
    return 'Give the sentiment and guidance direction of this as JSON:\n' + text

def evaluate_extraction(prompt_fn, label):
    valid = sent_ok = guid_ok = 0
    n = len(EXCERPTS)
    for ex in EXCERPTS:
        obj = parse_json(generate(prompt_fn(ex['text'])))
        if obj is None:
            continue
        valid += 1
        if str(obj.get('sentiment', '')).lower() == ex['gold']['sentiment']:
            sent_ok += 1
        if str(obj.get('guidance_direction', '')).lower() == ex['gold']['guidance_direction']:
            guid_ok += 1
    print(f'  {label:22s} valid {valid}/{n} | sentiment {sent_ok}/{n} | guidance {guid_ok}/{n}')
    return valid, sent_ok, guid_ok

print('Your structured + CoT prompt vs a minimal prompt:')
evaluate_extraction(build_extraction_prompt, 'yours (CoT + schema)')
evaluate_extraction(build_extraction_prompt_minimal, 'minimal')

Your structured + CoT prompt vs a minimal prompt:


#### ✍️ Reflect — what the numbers say
- Which prompt had the higher valid-JSON rate, and by how much? _…_
- Was `guidance_direction` agreement higher than `sentiment`? Why would objective fields extract better? _…_
- A downstream pipeline needs valid records. Which prompt would you ship, and what would you add to make parsing even more reliable? _…_

## 9. On-ramp to real 8-K data

The excerpts here are press-release-*style* text. The real target is **8-K EX-99.1 earnings releases** pulled from EDGAR — the heavier data pipeline described in `PROJECT_SCOPE.md` (SEC `submissions` API + `Archives`, respecting fair-access limits and a declared User-Agent). **Your extraction code doesn't change** — you'd feed it real filing text instead of these examples. That swap (clean examples → long, messy real filings) is exactly where structured extraction earns its keep: the validity rate drops, the figures get harder to pull, and the measurement harness you just wrote becomes the thing that tells you how well it's working at scale.

## 10. Recap & next

You turned the model into a structured extractor: a CoT + schema prompt, a robust JSON parser that fails gracefully, and a measured comparison that showed **validity is the gatekeeper** and **objective fields extract more reliably than subjective sentiment**. That record — not a bare label — is what real financial NLP systems actually consume.

**That completes Track B (Parts 5–7).** Next is the convergence:

**Part 8 — the bake-off:** one results table scoring *every* approach you've built — lexicon, fine-tuned GPT-2, BERT vs FinBERT, QLoRA Mistral, zero/few-shot LLM, synthetic-augmented — on a single held-out set. The portfolio centerpiece, and where the `Choosing_a_Training_Method` scoreboard finally fills in.